### Analyze SBB Schedule Using Graph Algorithms

We are going to employ the graph algorithms (BFS, shortest-path) to explore Swiss Railways schedules.

#### Notes
The code provides Swiss public transport stops as adjacency graph.

Get the graph:

In [1]:
import fahrplan
sbb = fahrplan.latest

Display some data:

In [2]:
len(sbb)

35515

In [3]:
sbb['Romanshorn']

{'Amriswil': 3,
 'Egnach': 2,
 'Konstanz': 15,
 'Kreuzlingen Hafen': 14,
 'Neukirch-Egnach': 2,
 'Romanshorn': 3,
 'Romanshorn (See)': 5,
 'Romanshorn Autoquai': 6,
 'Romanshorn, Bahnhof': 3,
 'St. Gallen': 18,
 'Uttwil': 3,
 'Wittenbach': 12}

The graph contains direct non-stop connections between stops, including transfers, with connection time in minutes.

The graph only records whether two stations are directly connected with each other by at least one connection. It omits any information on
  * lines
  * time of day
  * service schedules

So, some edges may be the consequence of special effects such as end-of-day connections, weekend night service, and the like.

Also, since each edge distance is recorded in minutes from departure to arrival, concatenating multiple legs will underestimate actual travelling time, as the stop time
is ignored.


### Run Shortest-Path

In [4]:
import graphs

path = graphs.shortest_path(sbb, 'Romanshorn', 'Bern')
path

{'path': [('Romanshorn', 0),
  ('Amriswil', 3),
  ('Oberaach', 2),
  ('Erlen', 1),
  ('Sulgen', 2),
  ('Weinfelden', 5),
  ('Märstetten', 2),
  ('Müllheim-Wigoltingen', 2),
  ('Hüttlingen-Mettendorf', 2),
  ('Felben-Wellhausen', 1),
  ('Frauenfeld', 2),
  ('Islikon', 3),
  ('Rickenbach-Attikon', 2),
  ('Wiesendangen', 0),
  ('Oberwinterthur', 1),
  ('Winterthur', 3),
  ('Kemptthal', 3),
  ('Zürich Flughafen', 8),
  ('Zürich Oerlikon', 3),
  ('Zürich Hardbrücke', 0),
  ('Zürich Altstetten', 3),
  ('Killwangen-Spreitenbach', 5),
  ('Mellingen Heitersberg', 5),
  ('Mägenwil', 2),
  ('Othmarsingen', 0),
  ('Lenzburg', 3),
  ('Aarau', 5),
  ('Olten', 8),
  ('Bern', 25)],
 'length': 101}

#### Visualize Stops

In [5]:
from fahrplan import gtfs_reader
# Read stop coordinates from GTFS.
stops = gtfs_reader.read_stops('fahrplan/gtfs_fp2026_20260919/', 'name')

In [6]:
def read_stops(src, dest, algo=graphs.shortest_path):
    """Computes a path from src to dest using the given algorithm and returns a tuple
       for each stop containing (stop_name, latitude, longitude, minutes)."""
    path = algo(sbb, src, dest)
    # DFS only returns stops, but not additional information.
    if algo == graphs.find_path_dfs:
        for stop in path:
            info = stops[stop]
            yield(info.name, float(info.lat), float(info.lon), 0)
        return
    for stop in path['path']:
        info = stops[stop[0]]
        yield(info.name, float(info.lat), float(info.lon), stop[1])

list(read_stops('Romanshorn', 'Bern', graphs.find_path_bfs))

[('Romanshorn', 47.56552099, 9.37937276, 0),
 ('St. Gallen', 47.42317725, 9.36989554, 18),
 ('Winterthur', 47.50033307, 8.7238182, 32),
 ('Bern', 46.94883109, 7.43912853, 98)]

In [17]:
#%pip install folium pillow selenium
import folium
import folium.features


def drawPath(map, path, tooltip, color):
    """Draws a polyline of the path on the map."""
    path = list(path) # reify generator
    line = folium.features.PolyLine([(info[1], info[2]) for info in path], tooltip=tooltip, color=color).add_to(map)
    map.fit_bounds(line.get_bounds())
    for info in path:
        folium.CircleMarker(
            location=[info[1], info[2]],
            radius=4,
            color=color,
            stroke=False,
            fill=True,
            fill_opacity=0.5,
            opacity=1,
            popup=info[0],
            tooltip=info[0]
        ).add_to(map)


import os
# CARTO_API_KEY = os.getenv("CARTO_API_KEY", "secret key")

map = folium.Map(location=[47.5, 8.5])

# # Add CartoDB Positron with the API key appended
# folium.TileLayer(
#     tiles=(
#         "https://{s}.basemaps.cartocdn.com/rastertiles/positron/{z}/{x}/{y}.png?key="
#         + CARTO_API_KEY
#     ),
#     attr=(
#         '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>'
#         ' contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'
#     ),
#     subdomains="abcd",  # Required so Leaflet can resolve {s}
#     max_zoom=20,
# ).add_to(map)

import sys
sys.setrecursionlimit(10000)
origin = 'Romanshorn'
destination = 'Neukirch-Egnach'
drawPath(map, read_stops(origin, destination, graphs.shortest_path), "Dijkstra", "darkblue")
drawPath(map, read_stops(origin, destination, graphs.find_path_bfs), "BFS", "red")
drawPath(map, read_stops(origin, destination, graphs.find_path_dfs), "DFS", "green")

# map

# Export to still image in order for github to render.
# Folium interactive content cannot be rendered on github.
import io
from PIL import Image

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument('--headless=new')
driver = webdriver.Chrome(options=options)
# img_data = map._to_png(15, driver=driver)
# Image.open(io.BytesIO(img_data))

map